# EEG Emotional Memory Classification — Colab Pipeline v3.0

**Pipeline Steps:**
1. Install dependencies & import libraries
2. Configure paths (Google Drive or Kaggle)
3. Define HDF5 loader & preprocessing
4. Define feature extraction (spectral + connectivity)
5. Define CSP spatial filters
6. Define ensemble models
7. Define post-processing (smoothing + temporal gating)
8. Define LOSO cross-validation
9. Define submission generator

> ⚙️ Before running: mount your Google Drive and place data under `MyDrive/eeg_competition/`


## Cell 1 — Install Dependencies

In [ ]:
# Install extra packages not included in Colab by default
import subprocess, sys

pkgs = ['lightgbm', 'xgboost']
for pkg in pkgs:
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                            capture_output=True, text=True)
    print(f"✓ {pkg} installed" if result.returncode == 0 else f"✗ {pkg} failed: {result.stderr}")

print("\nDone installing.")


## Cell 2 — Imports

In [ ]:
import numpy as np
import pandas as pd
import h5py
import os, re, time, warnings
from pathlib import Path

from scipy.signal import butter, filtfilt, hilbert, detrend, savgol_filter
from scipy.ndimage import gaussian_filter1d
from scipy.stats import skew, kurtosis
from scipy.linalg import eigh

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import SelectKBest, f_classif

import lightgbm as lgb
import xgboost as xgb
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
np.random.seed(42)
print("✓ All imports successful")


## Cell 3 — Mount Google Drive & Configure Paths

Place your data in Google Drive under:
```
MyDrive/
  eeg_competition/
    training/
      sleep_emo/   ← emotional .mat files
      sleep_neu/   ← neutral .mat files
    testing/       ← test_subject_1.mat, test_subject_7.mat, test_subject_12.mat
```


In [ ]:
# ── Set paths ─────────────────────────────────────────────────────────────
DRIVE_BASE = 'D:\\eeg_competition'

EMO_DIR  = f'{DRIVE_BASE}/training/sleep_emo'
NEU_DIR  = f'{DRIVE_BASE}/training/sleep_neu'
TEST_DIR = f'{DRIVE_BASE}/testing'
OUTPUT   = '/content/submission.csv'    # saved locally in Colab session

# ── Verify paths ──────────────────────────────────────────────────────────
for name, path in [('EMO_DIR', EMO_DIR), ('NEU_DIR', NEU_DIR), ('TEST_DIR', TEST_DIR)]:
    exists = os.path.exists(path)
    count  = len(list(Path(path).glob('*.mat'))) if exists else 0
    status = f'✓  ({count} .mat files)' if exists else '✗  NOT FOUND — check your Drive path'
    print(f'{name:12s}: {path}')
    print(f'               {status}')
    print()


## Cell 4 — EEG Constants & Hyperparameters

In [ ]:
# ── EEG constants (confirmed from EDA) ────────────────────────────────────
FS       = 200      # Hz
N_TP     = 200      # timepoints per trial (0–1 s at 200 Hz)
N_CH     = 16       # EEG channels

# ── Critical timing window (EDA finding #3) ───────────────────────────────
# Emotional signal emerges at ~300ms; predictions outside are blended to 0.5
T_START_MS  = 300
T_END_MS    = 900
TP_START    = int(T_START_MS / 1000 * FS)   # = 60
TP_END      = int(T_END_MS   / 1000 * FS)   # = 180
BLEND_ALPHA = 0.40   # outside window: blend weight toward 0.5

# ── Feature extraction ────────────────────────────────────────────────────
WIN          = 40        # sliding context window (200 ms)
N_CSP        = 4         # CSP filters per side (total = 2*N_CSP = 8)
N_FEAT_SEL   = 200       # ANOVA top-K features per fold (None = keep all)
SMOOTH_SIGMA = 8         # Gaussian σ for temporal smoothing
SAVGOL_WIN   = 21        # Savitzky-Golay window (must be odd)
SAVGOL_POLY  = 3

# ── Channel map ───────────────────────────────────────────────────────────
CHANNELS = ['c3','c4','o1','o2','cp3','f3','f4','cp4',
            'c5','cz','c6','cp5','p7','pz','p8','cp6']
CH = {c: i for i, c in enumerate(CHANNELS)}

# ── Frequency bands ───────────────────────────────────────────────────────
BANDS = {
    'delta': (0.5,  4.0),
    'theta': (4.0,  8.0),
    'alpha': (8.0,  13.0),
    'sigma': (12.0, 16.0),   # sleep spindles
    'beta':  (13.0, 30.0),
    'hbeta': (20.0, 40.0),
}

# ── Connectivity channel pairs (frontal–parietal) ─────────────────────────
CONN_PAIRS = [
    ('f3','pz'), ('f4','pz'), ('f3','cz'), ('f4','cz'),
    ('c3','c4'), ('cp3','cp4'), ('f3','f4'), ('cz','pz'),
]

print(f"✓ Config loaded")
print(f"  Emotional window : [{TP_START}:{TP_END}] timepoints  ({T_START_MS}–{T_END_MS} ms)")
print(f"  CSP components   : {2*N_CSP}")
print(f"  Feature selection: top {N_FEAT_SEL}")


## Cell 5 — Robust HDF5 / MATLAB Loader

In [ ]:
def _resolve_field(f, grp, key):
    """Safely resolve a field from a MATLAB HDF5 group (handles h5py.Reference)."""
    field = grp[key]
    if isinstance(field, h5py.Dataset):
        val = field[()]
        if isinstance(val, h5py.Reference):
            return np.array(f[val])
        if hasattr(val, 'shape') and val.shape == (1, 1):
            ref = val.item()
            if isinstance(ref, h5py.Reference):
                return np.array(f[ref])
        return np.array(val)
    return np.array(field)


def load_mat(path: str) -> dict:
    """
    Load one subject .mat file (MATLAB v7.3 / HDF5 format).
    Returns dict: eeg (n_trials,16,200), labels (n_trials,), time (200,).
    """
    path = str(path)
    with h5py.File(path, 'r') as f:
        # Navigate to the 'data' struct
        if 'data' in f:
            grp = f['data']
        else:
            grp = None
            for k in f.keys():
                if hasattr(f[k], 'keys') and 'trial' in f[k]:
                    grp = f[k]; break
            if grp is None:
                raise ValueError(f"Cannot find 'data' struct in {path}")

        # ── EEG trials ────────────────────────────────────────────────
        trial_raw = _resolve_field(f, grp, 'trial')
        if trial_raw.ndim == 3:
            sh = trial_raw.shape
            if sh[2] == N_CH and sh[1] == N_TP:
                trial_raw = trial_raw.transpose(0, 2, 1)
            elif sh[0] == N_CH and sh[1] == N_TP:
                trial_raw = trial_raw.transpose(2, 0, 1)
            elif sh[0] == N_TP and sh[1] == N_CH:
                trial_raw = trial_raw.transpose(2, 1, 0)
            # else already (n_trials, 16, 200)
        elif trial_raw.ndim == 2:
            trial_raw = trial_raw.T[np.newaxis]
        eeg = trial_raw.astype(np.float32)

        # ── Labels ────────────────────────────────────────────────────
        try:
            ti = _resolve_field(f, grp, 'trialinfo')
            if ti.ndim == 2:
                if ti.shape[0] == eeg.shape[0]:
                    labels = ti[:, 0].astype(int)
                elif ti.shape[1] == eeg.shape[0]:
                    labels = ti[0, :].astype(int)
                else:
                    labels = ti.flatten()[:eeg.shape[0]].astype(int)
            else:
                labels = ti.flatten().astype(int)
        except Exception:
            labels = np.ones(eeg.shape[0], dtype=int)

        # ── Time vector ───────────────────────────────────────────────
        try:
            tv = _resolve_field(f, grp, 'time').flatten()
        except Exception:
            tv = np.arange(N_TP) / FS

        # Keep only post-cue timepoints (t >= 0)
        t_mask = tv >= -1e-6
        if np.any(~t_mask):
            tv  = tv[t_mask]
            eeg = eeg[:, :, t_mask]
        if len(tv) != N_TP:
            tv = np.arange(N_TP) / FS

    print(f"    ✓ eeg={eeg.shape} | neu={(labels==1).sum()} emo={(labels==2).sum()}")
    return {'eeg': eeg, 'labels': labels, 'time': tv}


def load_all_training(emo_dir, neu_dir):
    """Load all training subjects from sleep_emo/ and sleep_neu/ folders."""
    subjects, seen = [], set()
    for folder in [emo_dir, neu_dir]:
        for fpath in sorted(Path(folder).glob('*.mat')):
            stem = fpath.stem
            if stem in seen: continue
            seen.add(stem)
            print(f"  Loading {fpath.name} ...")
            try:
                d = load_mat(str(fpath))
                d['id'] = stem; d['path'] = str(fpath)
                subjects.append(d)
            except Exception as e:
                print(f"    ✗ {e}")
    print(f"\n✓ Training: {len(subjects)} subjects loaded")
    return subjects


def load_all_test(test_dir):
    """Load test subjects; numeric subject ID extracted from filename."""
    subjects = []
    for fpath in sorted(Path(test_dir).glob('*.mat')):
        print(f"  Loading {fpath.name} ...")
        try:
            d = load_mat(str(fpath))
            nums = re.findall(r'\d+', fpath.stem)
            d['id']      = fpath.stem
            d['subj_id'] = int(nums[-1]) if nums else (len(subjects)+1)
            d['path']    = str(fpath)
            subjects.append(d)
        except Exception as e:
            print(f"    ✗ {e}")
    print(f"\n✓ Test: {len(subjects)} subjects loaded")
    print(f"  Subject IDs for submission: {[s['subj_id'] for s in subjects]}")
    return subjects


print("✓ Data loading functions defined")


## Cell 6 — Preprocessing

In [ ]:
def bandpass(data, lo, hi, fs=FS, order=4):
    """Zero-phase Butterworth bandpass filter."""
    nyq  = fs / 2.0
    lo_  = max(lo / nyq, 1e-4)
    hi_  = min(hi / nyq, 0.9999)
    b, a = butter(order, [lo_, hi_], btype='band')
    return filtfilt(b, a, data, axis=-1).astype(np.float32)


def preprocess_trial(raw_trial):
    """
    Single-trial preprocessing (EDA-validated):
      1. Detrend (remove linear DC drift)
      2. Average-reference
      3. Bandpass 0.5–40 Hz  ← EDA finding #5
    raw_trial: (16, 200) float32
    """
    x = detrend(raw_trial.astype(np.float64), axis=-1).astype(np.float32)
    x = x - x.mean(axis=0, keepdims=True)
    x = bandpass(x, lo=0.5, hi=40.0, fs=FS, order=4)
    return x


def zscore_subject(eeg_trials):
    """
    Per-subject Z-score normalization  ← EDA finding #2.
    Normalizes over all trials×timepoints per channel.
    eeg_trials: (n_trials, 16, 200) → returns normalized, mu, sigma.
    """
    mu  = eeg_trials.mean(axis=(0, 2), keepdims=True)
    sig = eeg_trials.std( axis=(0, 2), keepdims=True) + 1e-8
    return ((eeg_trials - mu) / sig).astype(np.float32), mu, sig


def apply_zscore(eeg_trials, mu, sig):
    return ((eeg_trials - mu) / sig).astype(np.float32)


print("✓ Preprocessing functions defined")


## Cell 7 — Feature Utility Functions

In [ ]:
def differential_entropy(x):
    """DE = 0.5·log(2πe·σ²) — proven best for EEG emotion."""
    v = np.var(x)
    return float(0.5 * np.log(2.0 * np.pi * np.e * v)) if v > 1e-14 else 0.0


def hjorth(x):
    """Activity, Mobility, Complexity."""
    act = float(np.var(x))
    d1  = np.diff(x); v1 = float(np.var(d1))
    mob = np.sqrt(v1 / (act + 1e-12))
    d2  = np.diff(d1); v2 = float(np.var(d2))
    cmp = np.sqrt(v2 / (v1 + 1e-12)) / (mob + 1e-12)
    return act, mob, cmp


def plv(x, y):
    """Phase Locking Value between two signals."""
    phi_x = np.angle(hilbert(x))
    phi_y = np.angle(hilbert(y))
    return float(np.abs(np.mean(np.exp(1j * (phi_x - phi_y)))))


print("✓ Feature utility functions defined")


## Cell 8 — Full Feature Extraction (~399 features per timepoint)

In [ ]:
def extract_trial_features(trial, win=WIN):
    """
    Extract ~399 neuroscientific features per timepoint.
    trial  : (16, 200) preprocessed EEG
    returns: (200, n_features)

    Feature groups:
      A. Instantaneous band power per channel×band   (96)
      B. Differential Entropy per channel×band       (96)
      C. Frontal Alpha Asymmetry (FAA)               ( 1)
      D. Frontal theta power                         ( 1)
      E. Theta/Beta ratio per channel                (16)
      F. Theta/Alpha ratio per channel               (16)
      G. Inter-hemispheric log-power asymmetry       ( 9)
      H. Hjorth (Activity, Mobility, Complexity)     (12)
      I. Sleep spindle power (sigma)                 ( 8)
      J. Relative band power per channel             (96)
      K. Peak-to-peak amplitude                      (16)
      L. Skewness + Kurtosis (key channels)          ( 8)
      M. Inter-channel PLV connectivity              (24)
    """
    n_ch, n_tp = trial.shape
    half = win // 2

    # Pre-compute band-filtered signals and Hilbert powers
    bf, bp = {}, {}
    for bname, (lo, hi) in BANDS.items():
        f         = bandpass(trial, lo, hi, FS)
        bf[bname] = f
        bp[bname] = (np.abs(hilbert(f, axis=-1)) ** 2).astype(np.float32)

    all_feats = []
    for t in range(n_tp):
        t0 = max(0, t - half); t1 = min(n_tp, t + half)
        f  = []

        # A. Instantaneous band power
        for bn in BANDS:
            f.extend(np.mean(bp[bn][:, t0:t1], axis=1).tolist())

        # B. Differential Entropy
        for bn in BANDS:
            seg = bf[bn][:, t0:t1]
            for ch in range(n_ch):
                f.append(differential_entropy(seg[ch]))

        # C. Frontal Alpha Asymmetry (FAA)
        f3a = np.mean(bp['alpha'][CH['f3'], t0:t1]) + 1e-12
        f4a = np.mean(bp['alpha'][CH['f4'], t0:t1]) + 1e-12
        f.append(float(np.log(f4a) - np.log(f3a)))

        # D. Frontal theta power
        f.append(float((np.mean(bp['theta'][CH['f3'], t0:t1]) +
                        np.mean(bp['theta'][CH['f4'], t0:t1])) / 2.0))

        # E. Theta/Beta per channel
        for ch in range(n_ch):
            th = np.mean(bp['theta'][ch, t0:t1]) + 1e-12
            be = np.mean(bp['beta'][ch,  t0:t1]) + 1e-12
            f.append(float(th / be))

        # F. Theta/Alpha per channel
        for ch in range(n_ch):
            th = np.mean(bp['theta'][ch, t0:t1]) + 1e-12
            al = np.mean(bp['alpha'][ch, t0:t1]) + 1e-12
            f.append(float(th / al))

        # G. Inter-hemispheric log-power asymmetry
        for ch1, ch2 in [('c3','c4'), ('cp3','cp4'), ('o1','o2')]:
            for bn in ['theta', 'alpha', 'beta']:
                p1 = np.mean(bp[bn][CH[ch1], t0:t1]) + 1e-12
                p2 = np.mean(bp[bn][CH[ch2], t0:t1]) + 1e-12
                f.append(float(np.log(p2) - np.log(p1)))

        # H. Hjorth parameters
        for chn in ['f3', 'f4', 'cz', 'pz']:
            act, mob, cmp = hjorth(trial[CH[chn], t0:t1])
            f.extend([act, mob, cmp])

        # I. Sleep spindle power (sigma band)
        for chn in ['c3','c4','cz','pz','f3','f4','cp3','cp4']:
            f.append(float(np.mean(bp['sigma'][CH[chn], t0:t1])))

        # J. Relative band power
        total = sum(np.mean(bp[bn][:, t0:t1], axis=1) for bn in BANDS) + 1e-12
        for bn in BANDS:
            f.extend((np.mean(bp[bn][:, t0:t1], axis=1) / total).tolist())

        # K. Peak-to-peak amplitude
        seg = trial[:, t0:t1]
        f.extend((np.max(seg, axis=1) - np.min(seg, axis=1)).tolist())

        # L. Skewness + Kurtosis (key channels)
        for chn in ['f3', 'f4', 'cz', 'pz']:
            s = trial[CH[chn], t0:t1].astype(np.float64)
            f.append(float(skew(s))     if len(s) > 2 else 0.0)
            f.append(float(kurtosis(s)) if len(s) > 3 else 0.0)

        # M. Inter-channel PLV connectivity
        for bn in ['theta', 'alpha', 'beta']:
            for ch1, ch2 in CONN_PAIRS:
                s1 = bf[bn][CH[ch1], t0:t1]
                s2 = bf[bn][CH[ch2], t0:t1]
                f.append(plv(s1, s2) if len(s1) >= 4 else 0.0)

        all_feats.append(f)

    F = np.array(all_feats, dtype=np.float32)
    F = np.nan_to_num(F, nan=0.0, posinf=0.0, neginf=0.0)
    return F   # (200, n_features)


def extract_subject_features(subj_dict, zmu=None, zsig=None):
    """
    Full feature extraction for one subject.
    If zmu/zsig None → fit per-subject Z-score (EDA finding #2).
    Returns X (n_trials*200, n_features), y, trial_ids, zmu, zsig.
    """
    eeg    = subj_dict['eeg'].copy()
    labels = subj_dict['labels']

    if zmu is None:
        eeg, zmu, zsig = zscore_subject(eeg)
    else:
        eeg = apply_zscore(eeg, zmu, zsig)

    n_trials = eeg.shape[0]
    feats_list, labels_list, trial_list = [], [], []

    for i in range(n_trials):
        trial = preprocess_trial(eeg[i])
        F     = extract_trial_features(trial)
        feats_list.append(F)
        labels_list.extend([int(labels[i])] * N_TP)
        trial_list.extend([i] * N_TP)

    X = np.vstack(feats_list)
    y = np.array(labels_list, dtype=np.int32)
    t = np.array(trial_list,  dtype=np.int32)
    return X, y, t, zmu, zsig


print("✓ Feature extraction functions defined")


## Cell 9 — CSP Spatial Filters

In [ ]:
class CSP:
    """
    Common Spatial Patterns — maximizes class variance separation.
    Must be fitted on training subjects ONLY (no leakage).
    """
    def __init__(self, n=N_CSP, band_lo=4.0, band_hi=8.0):
        self.n = n; self.lo = band_lo; self.hi = band_hi; self.W = None

    def fit(self, eeg, labels):
        """eeg: (n_trials, 16, 200), labels: (n_trials,) — values 1 or 2"""
        filtered = bandpass(eeg.reshape(-1, N_TP), self.lo, self.hi, FS
                            ).reshape(eeg.shape)
        def cov(X):
            C = np.zeros((N_CH, N_CH))
            for t in range(len(X)):
                s = X[t] - X[t].mean(axis=-1, keepdims=True)
                C += s @ s.T / (s.shape[-1] - 1)
            return C / len(X)
        C1 = cov(filtered[labels == 1])
        C2 = cov(filtered[labels == 2])
        ev, evec = eigh(C1, C1 + C2)
        idx = np.argsort(ev)
        sel = np.concatenate([idx[:self.n], idx[-self.n:]])
        self.W = evec[:, sel]   # (16, 2*n)
        return self

    def transform(self, eeg):
        """eeg: (n_trials, 16, 200) → (n_trials, 2*n, 200)"""
        filtered = bandpass(eeg.reshape(-1, N_TP), self.lo, self.hi, FS
                            ).reshape(eeg.shape)
        return np.tensordot(self.W.T, filtered, axes=([1],[1])).transpose(1,0,2)

    def log_var_features(self, eeg, win=WIN):
        """Log-variance of CSP signals in sliding window → (n_trials*N_TP, 2*n)"""
        csp_sig = self.transform(eeg)
        half = win // 2
        n_tr, n_csp, n_tp = csp_sig.shape
        out = []
        for tr in range(n_tr):
            for t in range(n_tp):
                t0 = max(0, t-half); t1 = min(n_tp, t+half)
                lv = np.log(np.var(csp_sig[tr, :, t0:t1], axis=1) + 1e-12)
                out.append(lv)
        return np.array(out, dtype=np.float32)


print("✓ CSP class defined")


## Cell 10 — Ensemble Models (LDA + LightGBM + XGBoost + LR)

In [ ]:
def make_lda():
    return LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')

def make_lgbm():
    return lgb.LGBMClassifier(
        n_estimators=600, num_leaves=31, max_depth=6,
        learning_rate=0.04, subsample=0.8, colsample_bytree=0.7,
        min_child_samples=20, class_weight='balanced',
        reg_alpha=0.1, reg_lambda=1.0,
        n_jobs=-1, random_state=42, verbose=-1)

def make_xgb():
    return xgb.XGBClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_weight=10,
        gamma=0.1, eval_metric='logloss',
        n_jobs=-1, random_state=42, verbosity=0)

def make_lr():
    return LogisticRegression(
        C=0.1, max_iter=500, class_weight='balanced', random_state=42)


def train_predict(X_tr, y_tr, X_te, k=N_FEAT_SEL):
    """
    Fit ensemble on training data, predict test data.
    Returns raw probability array (length = len(X_te)).
    """
    y_bin = (y_tr == 2).astype(int)

    # Feature selection (per fold — no leakage)
    if k and k < X_tr.shape[1]:
        sel    = SelectKBest(f_classif, k=k)
        X_tr_s = sel.fit_transform(X_tr, y_bin)
        X_te_s = sel.transform(X_te)
    else:
        X_tr_s, X_te_s = X_tr, X_te

    # RobustScaler (handles EEG outliers well)
    sc     = RobustScaler()
    X_tr_s = sc.fit_transform(X_tr_s)
    X_te_s = sc.transform(X_te_s)

    probs = np.zeros(len(X_te_s), dtype=np.float64)

    lda = make_lda(); lda.fit(X_tr_s, y_bin)
    probs += 0.25 * lda.predict_proba(X_te_s)[:, 1]

    lgbm = make_lgbm(); lgbm.fit(X_tr_s, y_bin)
    probs += 0.30 * lgbm.predict_proba(X_te_s)[:, 1]

    xgbm = make_xgb(); xgbm.fit(X_tr_s, y_bin)
    probs += 0.25 * xgbm.predict_proba(X_te_s)[:, 1]

    lr = make_lr(); lr.fit(X_tr_s, y_bin)
    probs += 0.20 * lr.predict_proba(X_te_s)[:, 1]

    return probs


print("✓ Ensemble model functions defined")


## Cell 11 — Post-Processing (Smoothing + Temporal Gating)

In [ ]:
def smooth_predictions(raw_probs, trial_ids,
                       sigma=SMOOTH_SIGMA,
                       sg_win=SAVGOL_WIN, sg_poly=SAVGOL_POLY):
    """
    Two-stage per-trial smoothing:
      1. Savitzky-Golay  — preserves shape, removes HF noise
      2. Gaussian        — enforces temporal continuity (rewards sustained AUC)
    """
    smoothed = np.zeros_like(raw_probs, dtype=np.float64)
    for tid in np.unique(trial_ids):
        mask = (trial_ids == tid)
        seg  = raw_probs[mask].astype(np.float64)
        if len(seg) >= sg_win:
            seg = savgol_filter(seg, window_length=sg_win, polyorder=sg_poly)
        seg = gaussian_filter1d(seg, sigma=sigma)
        smoothed[mask] = seg
    return smoothed


def temporal_gate(probs, trial_ids,
                  tp_start=TP_START, tp_end=TP_END,
                  alpha=BLEND_ALPHA):
    """
    EDA finding #3: emotional signal emerges after 300 ms.
    Outside [tp_start, tp_end]: blend prediction toward 0.5.
      pred_final = alpha * pred_model + (1-alpha) * 0.5   (outside window)
      pred_final = pred_model                              (inside  window)
    """
    gated = probs.copy()
    for tid in np.unique(trial_ids):
        mask = (trial_ids == tid)
        seg  = gated[mask]
        for tp in range(N_TP):
            if tp < tp_start or tp > tp_end:
                seg[tp] = alpha * seg[tp] + (1 - alpha) * 0.5
        gated[mask] = seg
    return gated


print("✓ Post-processing functions defined")


## Cell 12 — LOSO Cross-Validation

> ⏱️ **This step takes ~20–40 minutes** depending on Colab hardware.  
> Skip to Cell 13 if you only want the submission file.


In [ ]:
def run_loso(train_subjects):
    """
    Leave-One-Subject-Out CV:
      • Per-subject Z-score fitted on training set only
      • CSP fitted on training set only
      • Feature selection fitted on training set only (no leakage)
    """
    print("\n" + "="*62)
    print("  LOSO Cross-Validation")
    print("="*62)

    print("\nStep 1/2: Extracting features for all training subjects...")
    cache = []   # (X, y, t, eeg_raw, labels_raw, zmu, zsig)
    for i, s in enumerate(train_subjects):
        t0 = time.time()
        X, y, t, zmu, zsig = extract_subject_features(s)
        cache.append((X, y, t, s['eeg'], s['labels'], zmu, zsig))
        print(f"  [{i+1:2d}/{len(train_subjects)}] {s['id']}: "
              f"shape={X.shape}  ({time.time()-t0:.1f}s)")

    print("\nStep 2/2: LOSO folds...")
    aucs = []
    for val_i in range(len(train_subjects)):
        Xv, yv, tidv, eeg_v, lbl_v, _, _ = cache[val_i]

        X_tr      = np.vstack([cache[i][0] for i in range(len(cache)) if i != val_i])
        y_tr      = np.concatenate([cache[i][1] for i in range(len(cache)) if i != val_i])
        eeg_tr_all = np.vstack([cache[i][3] for i in range(len(cache)) if i != val_i])
        lbl_tr_all = np.concatenate([cache[i][4] for i in range(len(cache)) if i != val_i])

        csp = CSP(); csp.fit(eeg_tr_all, lbl_tr_all)
        csp_tr    = csp.log_var_features(eeg_tr_all)
        csp_v     = csp.log_var_features(eeg_v)
        X_tr_full = np.hstack([X_tr, csp_tr])
        Xv_full   = np.hstack([Xv,   csp_v])

        raw   = train_predict(X_tr_full, y_tr, Xv_full)
        sm    = smooth_predictions(raw, tidv)
        gated = temporal_gate(sm, tidv)
        gated = np.clip(gated, 0.01, 0.99)

        y_bin = (yv == 2).astype(int)
        auc   = roc_auc_score(y_bin, gated)
        aucs.append(auc)
        print(f"  Fold {val_i+1:2d} — {train_subjects[val_i]['id']}: "
              f"AUC = {auc:.4f}  {'↑ above chance' if auc > 0.5 else '↓ below chance'}")

    print(f"\n  Mean LOSO AUC : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
    print(f"  Best          : {max(aucs):.4f}")
    print(f"  Worst         : {min(aucs):.4f}")

    # Plot LOSO results
    plt.figure(figsize=(8, 4))
    plt.bar(range(1, len(aucs)+1), aucs, color=['green' if a>0.5 else 'red' for a in aucs])
    plt.axhline(0.5, color='black', linestyle='--', label='Chance (0.5)')
    plt.axhline(np.mean(aucs), color='blue', linestyle='-', label=f'Mean={np.mean(aucs):.3f}')
    plt.xlabel('Subject fold'); plt.ylabel('AUC')
    plt.title('LOSO Cross-Validation AUC per Subject')
    plt.legend(); plt.tight_layout(); plt.show()

    return aucs


print("✓ LOSO function defined")


## Cell 13 — Submission Generator

In [ ]:
def generate_submission(train_subjects, test_subjects, output_path=OUTPUT):
    """
    Train on ALL training data, predict 3 test subjects, save CSV.
    Submission ID format: {subject_id}_{trial}_{timepoint}
    """
    print("\n" + "="*62)
    print("  Final Training on ALL Data")
    print("="*62)

    # Step 1: Extract all training features
    print("\nExtracting training features...")
    X_all, y_all, eeg_all, lbl_all = [], [], [], []
    for i, s in enumerate(train_subjects):
        print(f"  [{i+1}/{len(train_subjects)}] {s['id']} ...", end=' ')
        X, y, _, zmu, zsig = extract_subject_features(s)
        s['_zmu'] = zmu; s['_zsig'] = zsig
        X_all.append(X); y_all.append(y)
        eeg_all.append(s['eeg']); lbl_all.append(s['labels'])
        print(f"shape={X.shape}")

    X_tr   = np.vstack(X_all)
    y_tr   = np.concatenate(y_all)
    eeg_tr = np.vstack(eeg_all)
    lbl_tr = np.concatenate(lbl_all)
    y_bin  = (y_tr == 2).astype(int)
    print(f"\n  Training matrix : {X_tr.shape}  "
          f"[emo={y_bin.sum():,} neu={(1-y_bin).sum():,}]")

    # Step 2: Fit CSP on all training EEG
    print("\nFitting CSP on full training set...")
    csp    = CSP(); csp.fit(eeg_tr, lbl_tr)
    csp_tr = csp.log_var_features(eeg_tr)
    X_tr_full = np.hstack([X_tr, csp_tr])
    print(f"  Full feature matrix: {X_tr_full.shape}")

    # Step 3: Feature selection + scale + train ensemble
    if N_FEAT_SEL and N_FEAT_SEL < X_tr_full.shape[1]:
        sel    = SelectKBest(f_classif, k=N_FEAT_SEL)
        X_tr_s = sel.fit_transform(X_tr_full, y_bin)
    else:
        sel    = None
        X_tr_s = X_tr_full

    sc     = RobustScaler()
    X_tr_s = sc.fit_transform(X_tr_s)

    print("\nTraining ensemble models...")
    lda  = make_lda();  lda.fit(X_tr_s, y_bin);  print("  ✓ LDA")
    lgbm = make_lgbm(); lgbm.fit(X_tr_s, y_bin); print("  ✓ LightGBM")
    xgbm = make_xgb();  xgbm.fit(X_tr_s, y_bin); print("  ✓ XGBoost")
    lr   = make_lr();   lr.fit(X_tr_s, y_bin);   print("  ✓ LogisticReg")

    # Step 4: Predict each test subject
    print("\nGenerating test predictions...")
    rows = []
    for s in test_subjects:
        subj_id  = s['subj_id']
        n_trials = s['eeg'].shape[0]
        print(f"\n  Test subject {subj_id} ({s['id']}): {n_trials} trials")

        X_te, _, tidv, _, _ = extract_subject_features(s)
        csp_te    = csp.log_var_features(s['eeg'])
        X_te_full = np.hstack([X_te, csp_te])

        X_te_s = sel.transform(X_te_full) if sel is not None else X_te_full
        X_te_s = sc.transform(X_te_s)

        raw_probs = (0.25 * lda.predict_proba(X_te_s)[:, 1] +
                     0.30 * lgbm.predict_proba(X_te_s)[:, 1] +
                     0.25 * xgbm.predict_proba(X_te_s)[:, 1] +
                     0.20 * lr.predict_proba(X_te_s)[:, 1])

        smoothed = smooth_predictions(raw_probs, tidv)
        gated    = temporal_gate(smoothed, tidv)
        gated    = np.clip(gated, 0.01, 0.99)

        ptr = 0
        for trial_id in range(n_trials):
            for tp in range(N_TP):
                rows.append({'id': f"{subj_id}_{trial_id}_{tp}",
                             'prediction': float(gated[ptr])})
                ptr += 1

        print(f"    pred range : [{gated.min():.3f}, {gated.max():.3f}]")
        print(f"    pred mean  : {gated.mean():.4f}")

    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False)
    print(f"\n{'='*62}")
    print(f"✓ Submission saved → {output_path}")
    print(f"  Total rows : {len(df):,}")
    print(df.head(6).to_string(index=False))
    return df


print("✓ Submission generator defined")


## ▶ Cell 14 — MAIN: Run Everything

> This cell runs the full pipeline:
> 1. Load training & test data
> 2. LOSO cross-validation (skip by setting `RUN_LOSO = False`)
> 3. Train final model on all data
> 4. Generate & download `submission.csv`


In [ ]:
# ── Toggle options ────────────────────────────────────────────────────────
RUN_LOSO = False   # Set True to run LOSO CV (~20-40 min); False to skip

# ─────────────────────────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════╗
║   EEG Emotional Memory — Colab Pipeline v3.0                ║
╠══════════════════════════════════════════════════════════════╣
║   EDA insights applied:                                      ║
║   • Per-subject Z-score normalisation                        ║
║   • 0.5–40 Hz bandpass                                       ║
║   • Temporal gating: focus on [300ms–900ms]                  ║
║   • Relative features: FAA, Theta/Beta, Asymmetry            ║
║   • CSP spatial filters (no leakage)                         ║
║   • LDA + LightGBM + XGBoost + LR ensemble                   ║
╚══════════════════════════════════════════════════════════════╝
""")

# STEP 1: Load training data
print("STEP 1: Loading training data...")
train_subjects = load_all_training(EMO_DIR, NEU_DIR)

# STEP 2: Load test data
print("\nSTEP 2: Loading test data...")
test_subjects = load_all_test(TEST_DIR)

# STEP 3: Optional LOSO CV
if RUN_LOSO:
    print("\nSTEP 3: LOSO Cross-Validation (~20–40 min)...")
    fold_aucs = run_loso(train_subjects)
    print(f"  Mean LOSO AUC: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")
else:
    print("\nSTEP 3: Skipping LOSO CV (set RUN_LOSO=True to enable)")
    fold_aucs = []

# STEP 4: Generate submission file
print("\nSTEP 4: Generating submission file...")
df = generate_submission(train_subjects, test_subjects, OUTPUT)

# STEP 5: Validate submission
print("\n" + "="*62)
print("STEP 5: Validating submission...")
assert 'id' in df.columns,         "FAIL: 'id' column missing"
assert 'prediction' in df.columns, "FAIL: 'prediction' column missing"
assert df['prediction'].between(0, 1).all(), "FAIL: predictions outside [0,1]"
sample_parts = str(df.iloc[0]['id']).split('_')
assert len(sample_parts) == 3, f"FAIL: ID format wrong: {df.iloc[0]['id']}"
print("✓ id column present")
print("✓ prediction column present")
print("✓ ID format correct (subject_trial_timepoint)")
print("✓ All predictions in [0, 1]")
print(f"✓ Total rows: {len(df):,}")

# STEP 6: Download submission.csv
print("\nSTEP 6: Downloading submission.csv...")
from google.colab import files
files.download(OUTPUT)

summary = f"Mean LOSO AUC: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}" if fold_aucs else "LOSO skipped"
print(f"""
╔══════════════════════════════════════════════════════════════╗
║  ✓ DONE!                                                    ║
╠══════════════════════════════════════════════════════════════╣
║  {summary:<58s}║
║  Submission : {OUTPUT:<46s}║
╚══════════════════════════════════════════════════════════════╝
""")


## Cell 15 — Optional: Diagnose a .mat File (if loading fails)

In [ ]:
def diagnose(path):
    """Print full HDF5 structure of a .mat file to debug loading issues."""
    print(f"\nDiagnosing: {path}")
    with h5py.File(path, 'r') as f:
        def show(name, obj):
            sh = obj.shape if hasattr(obj, 'shape') else 'group'
            dt = obj.dtype  if hasattr(obj, 'dtype') else '—'
            print(f"  {name:45s}  shape={sh}  dtype={dt}")
        f.visititems(show)

# ── Usage (uncomment and fill in path) ───────────────────────────────────
# diagnose(f'{EMO_DIR}/sub_01_sleep_emo.mat')
print("✓ diagnose() function ready — uncomment above to use")


## Cell 16 — Optional: Hyperparameter Tuning (run AFTER MAIN)

In [ ]:
def tune_smoothing(train_subjects):
    """Grid-search best smoothing sigma and Savitzky-Golay window via LOSO."""
    print("\nTuning smoothing parameters (quick LOSO, spectral features only)...")
    cache = []
    for s in train_subjects:
        X, y, t, _, _ = extract_subject_features(s)
        cache.append((X, y, t))

    best = {'sigma': SMOOTH_SIGMA, 'sg': SAVGOL_WIN, 'auc': 0.0}
    for sigma in [4, 6, 8, 10, 12]:
        for sgw in [11, 21, 31]:
            aucs = []
            for val_i in range(len(train_subjects)):
                Xv, yv, tidv = cache[val_i]
                X_tr = np.vstack([cache[i][0] for i in range(len(cache)) if i != val_i])
                y_tr = np.concatenate([cache[i][1] for i in range(len(cache)) if i != val_i])
                raw  = train_predict(X_tr, y_tr, Xv)
                sm   = smooth_predictions(raw, tidv, sigma=sigma, sg_win=sgw)
                gt   = temporal_gate(sm, tidv)
                aucs.append(roc_auc_score((yv==2).astype(int), np.clip(gt,0.01,0.99)))
            m = np.mean(aucs)
            print(f"  sigma={sigma:2d} sg_win={sgw:2d}: AUC={m:.4f}")
            if m > best['auc']:
                best.update({'sigma': sigma, 'sg': sgw, 'auc': m})

    print(f"\n  Best: SMOOTH_SIGMA={best['sigma']} SAVGOL_WIN={best['sg']} "
          f"(AUC={best['auc']:.4f})")
    return best

# ── Usage (uncomment after running MAIN) ─────────────────────────────────
# best_params = tune_smoothing(train_subjects)
print("✓ tune_smoothing() function ready — uncomment above to use")
